In [8]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import SGDClassifier


In [9]:
train_df = pd.read_csv("KDDTrain+.csv", header=None)
test_df = pd.read_csv("KDDTest+.csv", header=None)

test_df.columns = train_df.columns


In [10]:
drop_cols = [11, 13, 14, 20, 21]  # indices of useless NSL-KDD columns
train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)

X_train = train_df.iloc[:, :-1]
y_train = train_df.iloc[:, -1]
X_test = test_df.iloc[:, :-1]
y_test = test_df.iloc[:, -1]

In [11]:
le = LabelEncoder()

for col in X_train.columns:
    if X_train[col].dtype == "object":
        combined = pd.concat([X_train[col], X_test[col]])
        le.fit(combined)

        X_train[col] = le.transform(X_train[col])
        X_test[col] = le.transform(X_test[col])

if y_train.dtype == "object":
    le_y = LabelEncoder()
    combined_y = pd.concat([y_train, y_test])
    le_y.fit(combined_y)

    y_train = le_y.transform(y_train)
    y_test = le_y.transform(y_test)


In [14]:
print("TRAIN BASELINE")
print("Train samples:", len(train_df))
print("Train features:", X_train.shape[1])
print(y_train.value_counts())

print("\nTEST BASELINE")
print("Test samples:", len(test_df))
print("Test features:", X_test.shape[1])
print(y_test.value_counts())

TRAIN BASELINE
Train samples: 125973
Train features: 37
42
21    62557
18    20667
20    19339
19    10284
15     3990
17     3074
16     2393
12      729
14      674
11      641
13      451
10      253
9       194
7       118
8       106
6        96
5        81
4        79
0        66
3        65
1        62
2        54
Name: count, dtype: int64

TEST BASELINE
Test samples: 22543
Test features: 37
42
21    10694
18     2967
20     1343
15     1176
17     1168
19      890
14      736
16      681
13      519
12      486
11      461
7       249
10      195
6       156
8       131
0       123
3       116
9       106
5       103
4       101
1        87
2        55
Name: count, dtype: int64


In [12]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=300),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    "DecisionTree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier()
    }


In [13]:
for name, model in models.items():
    print("\n===============================")
    print(name)

    try:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, preds, average="macro", zero_division=0
        )
        weighted_f1 = precision_recall_fscore_support(
            y_test, preds, average="weighted", zero_division=0
        )[2]

        print(f"Accuracy: {acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"F1 Score(weighted): {weighted_f1:.4f}")

    except Exception as e:
        print("FAILED:", e)



RandomForest
Accuracy: 0.6432
Precision: 0.2377
Recall: 0.1727
F1 Score: 0.1721
F1 Score(weighted): 0.5915

XGBoost
Accuracy: 0.6406
Precision: 0.1909
Recall: 0.1850
F1 Score: 0.1776
F1 Score(weighted): 0.6110

DecisionTree
Accuracy: 0.6586
Precision: 0.2154
Recall: 0.1951
F1 Score: 0.1928
F1 Score(weighted): 0.6264

KNN
Accuracy: 0.6178
Precision: 0.1970
Recall: 0.1639
F1 Score: 0.1584
F1 Score(weighted): 0.5660
